## FRAP analyses - interpretation

___

In [ ]:
from pathlib import Path
from microlive.pipelines.pipeline_FRAP import *
from microlive.imports import *
from microlive import microscopy as mi
current_dir = Path().resolve()
one_drive_dir = mi.Utilities.get_one_drive_dir()

# Import plotting and data utility functions
from frap_utilities import (
    plot_FRAP_trajectories,
    plot_mean_trajectories_all,
    plot_box_swarm_final_values,
    plot_box_swarm_fit_results,
    load_frap_datasets,
    fit_all_cells,
)

In [ ]:
cwd = Path.cwd()
results_main_folder = cwd.joinpath('results')
results_folder = cwd.joinpath('results_interpretation')
results_folder.mkdir(parents=True, exist_ok=True)


In [ ]:

subfolder_strings = {'pNZ397' :  ["pNZ397", "None"] ,
                    'pNZ400': ["pNZ400", "None"]} 

list_datasets  = ['pNZ397','pNZ400']

datasets_to_process = ['pNZ397', 'pNZ400']

In [ ]:
frap_time = 10 # seconds

# Min–Max Normalization

We apply the following equation to normalize the values in the `mean_roi_frap_normalized` column:

$\text{normalized\_value} = \frac{\text{value} - \text{min\_val}}{\text{max\_val} - \text{min\_val}}$

In [ ]:
selected_field = "mean_roi_frap"

combined_df, total_number_cells = load_frap_datasets(
    results_main_folder=results_main_folder,
    subfolder_strings=subfolder_strings,
    list_datasets=list_datasets,
    selected_field=selected_field,
    apply_quality_check=True,
    drop_threshold=0.4,
    apply_min_max_normalization=True,
)

In [ ]:
df_all_fit_results = fit_all_cells(
    combined_df=combined_df,
    datasets_to_process=datasets_to_process,
    frap_time=frap_time,
    fit_function=fit_model_to_frap,
    selected_field=selected_field,
)

In [ ]:
# Group by dataset_type and count unique cell_id values
cell_counts = combined_df.groupby('dataset_type')['cell_id'].nunique()
# Print the total number of processed cells for each dataset type
for dataset_type, count in cell_counts.items():
    print(f"{dataset_type}: {count} cells processed")

In [ ]:
# Plot individual FRAP trajectories for each dataset
for ds in list_datasets:
    plot_FRAP_trajectories(df_list=combined_df, selected_dataset=ds, results_folder=results_folder)

In [ ]:
plot_mean_trajectories_all(combined_df, 
                            list_datasets,
                            selected_field='mean_roi_frap',
                            apply_quality_check=True,
                            drop_threshold=0.8,
                            apply_min_max_normalization=True,
                            # color_map = ['green' , 'darkgreen', 'black', 'grey'] ,
                            color_map = ['darkgreen', 'black'] ,
                            fig_size=(7,4),
                            use_sem=False, results_folder=results_folder)

In [ ]:
plot_box_swarm_final_values(
    df=combined_df,
    selected_field=selected_field,
    figsize=(4, 4),
    ylabel= "Normalized final recovery \n intensity",
    title="",
    y_min=0,
    #y_max=1.75,
    swarm_color="black",
    tick_size=14,
    order_categories = list_datasets, 
    show_stats=True,
    results_folder=results_folder,
)

In [ ]:
# Plot t_half_single comparison
plot_box_swarm_fit_results(
    df=df_all_fit_results,
    selected_field="t_half_single",
    figsize=(4, 4),
    ylabel=r"$t_{1/2}$ (s)",
    title="",
    y_min=0,
    order_categories =  list_datasets, 
    y_max=None,  # Let it auto-scale based on your data
    swarm_color="black",
    tick_size=14,
    show_stats=True,
    results_folder=results_folder,
)